In [ ]:
import sys

# Gerekli kutuphanelerin kurulumu
!{sys.executable} -m pip install torch torchvision torchaudio --quiet
!{sys.executable} -m pip install numpy matplotlib seaborn scikit-learn --quiet


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.facecolor'] = '#FAFAFA'

# XOR veri seti
girdi = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
hedef = np.array([[0], [1], [1], [0]], dtype=np.float32)

X_tensor = torch.FloatTensor(girdi)
y_tensor  = torch.FloatTensor(hedef)

etiketler_xor = ['(0,0)→0', '(0,1)→1', '(1,0)→1', '(1,1)→0']
renkler_nokta = ['#E74C3C', '#27AE60', '#27AE60', '#E74C3C']

print('XOR Dogruluk Tablosu:')
print(f"  {'A':^4} | {'B':^4} | {'A XOR B':^9}")
print('-' * 28)
for i in range(4):
    print(f"  {int(girdi[i,0]):^4} | {int(girdi[i,1]):^4} |    {int(hedef[i,0])}")


ModuleNotFoundError: No module named 'torch'

In [ ]:
# OR (dogrusal ayrilabilir) vs XOR (dogrusal AYRILAMAZ)
fig, eksenler = plt.subplots(1, 2, figsize=(12, 5))

hedef_or     = np.array([0, 1, 1, 1])
etiketler_or  = ['(0,0)→0', '(0,1)→1', '(1,0)→1', '(1,1)→1']
renkler_or    = ['#E74C3C' if l == 0 else '#27AE60' for l in hedef_or]
renkler_xor_v = ['#E74C3C' if l == 0 else '#27AE60' for l in hedef.flatten()]

for ax, baslik, cizgi_goster, renkler, etiketler in [
    (eksenler[0], 'OR Problemi (Dogrusal Ayrilabilir)',   True,  renkler_or,    etiketler_or),
    (eksenler[1], 'XOR Problemi (Dogrusal AYRILAMAZ!)',   False, renkler_xor_v, etiketler_xor)
]:
    ax.scatter(girdi[:, 0], girdi[:, 1], c=renkler, s=380,
               zorder=5, edgecolors='black', linewidth=2.5)
    for k, etiket in enumerate(etiketler):
        ax.annotate(etiket, (girdi[k, 0], girdi[k, 1]),
                    textcoords='offset points', xytext=(12, 10), fontsize=11)
    if cizgi_goster:
        xc = np.linspace(-0.3, 1.3, 100)
        ax.plot(xc, -xc + 0.5, 'b--', lw=2.5, label='Karar Siniri')
        ax.legend(fontsize=10, loc='upper left')
    else:
        ax.text(0.5, 0.5, 'Tek duz cizgi\nyetmez!',
                ha='center', va='center', fontsize=13, color='red', style='italic',
                bbox=dict(boxstyle='round', facecolor='#FFF3CD', alpha=0.8))
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('Giris A', fontsize=11)
    ax.set_ylabel('Giris B', fontsize=11)
    ax.set_title(baslik, fontsize=13, fontweight='bold')

fig.legend(handles=[mpatches.Patch(color='#E74C3C', label='Cikis = 0'),
                    mpatches.Patch(color='#27AE60', label='Cikis = 1')],
           loc='lower center', ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.04))
plt.suptitle('Sekil 1 — Dogrusal Ayrilabilirlik: OR vs XOR', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# --- Tek Katmanli Perceptron Denemesi (Basarisiz) ---
print('=' * 52)
print('  TEK KATMANLI PERCEPTRON  --  XOR DENEMESI')
print('  (Neden cozemedigini gostermek icin)')
print('=' * 52)

torch.manual_seed(7)

class TekKatmanPerceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.dogrusal = nn.Linear(2, 1)
        self.sigmoid  = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.dogrusal(x))

tek_model     = TekKatmanPerceptron()
tek_kayip_fn  = nn.BCELoss()
tek_optimizer = optim.Adam(tek_model.parameters(), lr=0.05)

tek_kayiplar = []
for epoch in range(2000):
    tek_model.train()
    tek_optimizer.zero_grad()
    cikti = tek_model(X_tensor)
    kayip = tek_kayip_fn(cikti, y_tensor)
    kayip.backward()
    tek_optimizer.step()
    tek_kayiplar.append(kayip.item())

tek_model.eval()
with torch.no_grad():
    tek_ham   = tek_model(X_tensor)
    tek_sinif = (tek_ham >= 0.5).float()

dogru_tek = int((tek_sinif == y_tensor).sum())
print(f'Son Kayip: {tek_kayiplar[-1]:.5f}')
print(f"\n  {'Giris':^10} | {'Gercek':^7} | {'Ham Cikis':^12} | {'Tahmin':^7}")
print('  ' + '-' * 46)
for i in range(4):
    print(f"  [{int(girdi[i,0])}, {int(girdi[i,1])}]      |    {int(hedef[i,0])}    "
          f"|   {tek_ham[i,0]:.5f}   |    {int(tek_sinif[i,0])}")
print(f'\nDogru: {dogru_tek}/4  ->  Basari: %{dogru_tek * 25:.0f}')
print('\n>>> Tek dogrusal katman XOR problemini ogrenemiyor! <<<')


In [ ]:
# --- Cok Katmanli Sinir Agi (2 gizli katman) ---
torch.manual_seed(99)

class XORSinirAgi(nn.Module):
    def __init__(self):
        super().__init__()
        self.gizli1  = nn.Linear(2, 6)
        self.gizli2  = nn.Linear(6, 6)
        self.cikis_k = nn.Linear(6, 1)
        self.tanh    = nn.Tanh()
        self.relu    = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.tanh(self.gizli1(x))
        x = self.relu(self.gizli2(x))
        x = self.sigmoid(self.cikis_k(x))
        return x

model       = XORSinirAgi()
kayip_fn    = nn.BCELoss()
optimizer   = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

EPOCH_SAYISI  = 8000
kayip_listesi = []
epoch_log     = []

for epoch in range(EPOCH_SAYISI):
    model.train()
    optimizer.zero_grad()
    cikti = model(X_tensor)
    kayip = kayip_fn(cikti, y_tensor)
    kayip.backward()
    optimizer.step()
    kayip_listesi.append(kayip.item())
    if (epoch + 1) % 1000 == 0:
        epoch_log.append((epoch + 1, kayip.item()))

model.eval()
with torch.no_grad():
    ham_cikti = model(X_tensor)
    tahmin    = (ham_cikti >= 0.5).float()

dogru_sayisi = int((tahmin == y_tensor).sum())

print('=' * 52)
print('   COK KATMANLI XOR SINIR AGI -- SONUCLAR')
print('=' * 52)
print(f'  Mimari      : 2 -> 6 (Tanh) -> 6 (ReLU) -> 1 (Sigmoid)')
print(f'  Optimizator : SGD  (lr=0.1, momentum=0.9)')
print(f'  Epoch       : {EPOCH_SAYISI}')
print(f'  Son Kayip   : {kayip_listesi[-1]:.6f}')
print()
print(f"  {'Giris':^10} | {'Gercek':^7} | {'Ham Cikis':^12} | {'Tahmin':^7}")
print('  ' + '-' * 46)
for i in range(4):
    print(f"  [{int(girdi[i,0])}, {int(girdi[i,1])}]      |    {int(hedef[i,0])}    "
          f"|   {ham_cikti[i,0]:.5f}   |    {int(tahmin[i,0])}")
print(f'\nDogru Tahmin: {dogru_sayisi}/4  ->  Basari: %{dogru_sayisi * 25:.0f}')
print('\n  Epoch / Kayip izleme:')
for ep, kp in epoch_log:
    print(f'    Epoch {ep:5d}: {kp:.6f}')


In [ ]:
fig, eksenler = plt.subplots(1, 2, figsize=(14, 5))

# Sol: iki modelin kayip egrisi karsilastirmasi
eksenler[0].plot(tek_kayiplar,  color='#E74C3C', lw=1.5, alpha=0.75, label='Tek Katmanli (basarisiz)')
eksenler[0].plot(kayip_listesi, color='#27AE60', lw=1.5, label='Cok Katmanli')
eksenler[0].set_title('Kayip Fonksiyonu Karsilastirmasi', fontsize=13, fontweight='bold')
eksenler[0].set_xlabel('Epoch', fontsize=11)
eksenler[0].set_ylabel('Binary Cross-Entropy Kaybi', fontsize=11)
eksenler[0].legend(fontsize=10)
eksenler[0].text(0.98, 0.95,
                 f'Cok katli son kayip: {kayip_listesi[-1]:.5f}',
                 transform=eksenler[0].transAxes, ha='right', va='top',
                 fontsize=9, color='#27AE60',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Sag: ogrenilen karar siniri
xx, yy = np.meshgrid(np.linspace(-0.3, 1.3, 400), np.linspace(-0.3, 1.3, 400))
izgara = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])
with torch.no_grad():
    zz = model(izgara).numpy().reshape(xx.shape)

cf = eksenler[1].contourf(xx, yy, zz, levels=50, cmap='coolwarm', alpha=0.7)
eksenler[1].contour(xx, yy, zz, levels=[0.5], colors='black', linewidths=2.5, linestyles='--')
plt.colorbar(cf, ax=eksenler[1], label='Model Ciktisi')

eksenler[1].scatter(girdi[:, 0], girdi[:, 1], c=renkler_nokta,
                    s=360, zorder=5, edgecolors='white', linewidth=2.5)
for i, etiket in enumerate(etiketler_xor):
    eksenler[1].annotate(etiket, (girdi[i, 0], girdi[i, 1]),
                         textcoords='offset points', xytext=(10, 8),
                         fontsize=10, color='black', fontweight='bold')

eksenler[1].set_title('Ogrenilen Karar Siniri (Cok Katmanli)', fontsize=13, fontweight='bold')
eksenler[1].set_xlabel('Giris A', fontsize=11)
eksenler[1].set_ylabel('Giris B', fontsize=11)

plt.suptitle('Sekil 2 — Egitim Sureci ve Karar Siniri', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

gercek     = hedef.flatten().astype(int)
tahmin_arr = tahmin.numpy().flatten().astype(int)

acc  = accuracy_score(gercek, tahmin_arr)
prec = precision_score(gercek, tahmin_arr, zero_division=0)
rec  = recall_score(gercek, tahmin_arr, zero_division=0)
f1   = f1_score(gercek, tahmin_arr, zero_division=0)
cm   = confusion_matrix(gercek, tahmin_arr)

print('PERFORMANS DEGERLENDIRMESI')
print(f'  Accuracy  (Dogruluk)  : {acc * 100:.1f}%')
print(f'  Precision (Kesinlik)  : {prec:.4f}')
print(f'  Recall    (Duyarlilik): {rec:.4f}')
print(f'  F1-Score              : {f1:.4f}')
print()
print('  Confusion Matrix:')
print(f'    TN={cm[0,0]:2d}  FP={cm[0,1]:2d}')
print(f'    FN={cm[1,0]:2d}  TP={cm[1,1]:2d}')


In [ ]:
fig, eksenler = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=eksenler[0],
            xticklabels=['Tahmin: 0', 'Tahmin: 1'],
            yticklabels=['Gercek: 0', 'Gercek: 1'],
            annot_kws={'size': 18, 'weight': 'bold'},
            linewidths=2, linecolor='white')
eksenler[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

metrik_isimleri  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrik_degerleri = [acc, prec, rec, f1]
renkler_bar      = ['#3498DB', '#2ECC71', '#E67E22', '#9B59B6']

barlar = eksenler[1].barh(metrik_isimleri, metrik_degerleri,
                           color=renkler_bar, edgecolor='white', linewidth=1.5, height=0.5)
for bar, deger in zip(barlar, metrik_degerleri):
    eksenler[1].text(deger + 0.01, bar.get_y() + bar.get_height() / 2,
                     f'{deger:.4f}', va='center', fontsize=12, fontweight='bold')
eksenler[1].set_xlim(0, 1.2)
eksenler[1].set_title('Performans Metrikleri', fontsize=13, fontweight='bold')
eksenler[1].set_xlabel('Deger', fontsize=11)
eksenler[1].axvline(x=1.0, color='red', linestyle='--', alpha=0.4, label='Maks. (1.0)')
eksenler[1].legend(fontsize=9)

plt.suptitle('Sekil 3 — Model Performans Degerlendirmesi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
